# 1A pilot v7 — CPU re-analyses for review 1

> **Tóm tắt (VI):** Notebook này **không chạy mô hình**. Nó đọc kết quả v4, v5 và v6 trên Drive và làm lại các phân tích reviewer yêu cầu. Chạy trên **CPU runtime**, `Run all`, cho phép Drive; mất vài phút. Kết quả ghi vào `1A-pilot-v7-cpu/` và không ghi đè gì của v4–v6. Bước C tạo file `natural_sample_100.csv` để bạn thêm dấu tay.

| # | Review point | What this notebook does |
|---|---|---|
| B1 | M4: few clusters | Size trend and adapted-vs-general test with p-values from t(G−1) and a wild cluster bootstrap (restricted, Webb weights); refits without the 14B models, without Phi, and with only families of ≥ 3 sizes |
| B2 | 5c, Q7: prior from noisy inputs | Batch calibration with a prior estimated on each condition's own (unlabelled) test inputs: drop, recovery and label collapse, next to v5's restoration on the same pairs |
| B3 | 6a, Q9: natural text | Balanced accuracy on naturally unaccented ViHSD comments vs v4's accented ViHSD comments (clean and stripped), per model |
| B4 | 7: restoration below clean? | 14B models: accuracy after strip → restore vs clean, paired bootstrap |
| B6 | Q11, Q2, M1 | Δ broken without the tokenizers that split *fewer* syllables after stripping; ΔNLL per token instead of per clean syllable; the ambiguity regression reported as its own specification |
| B7 | P1: figures | Dose-response (3-seed mean) and size figures redrawn from the final data |
| C | 6b, Q9 | Stratified sample of 100 naturally unaccented comments with the restorer's output, for manual diacritization |
| C2 | 6b, Q9 | From the annotated sample: restorer word error against the manual diacritization, and B3 re-scored without the comments whose label the annotator disputes, and with the annotator's labels |

## Setup and inputs

In [ ]:
import sys, os, re, json, unicodedata, itertools, warnings
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
SMOKE = bool(os.environ.get("PILOT_SMOKE"))   # local test only
SUF = "-smoke" if SMOKE else ""
SRC4, SRC5, SRC6 = (f"{ROOT}/1A-pilot-v{v}{SUF}" for v in (4, 5, 6))
OUT = f"{ROOT}/1A-pilot-v7-cpu{SUF}"
for d in ["figures", "tables", "data"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy
from IPython.display import display
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("reading", SRC4, SRC5, SRC6, "→ writing", OUT)

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)[:300]})
    print(("✅ " if ok else "❌ ") + name + (f" — {detail}" if detail else ""))
    pd.DataFrame(CHECKS).drop_duplicates("check", keep="last").to_csv(f"{OUT}/checks.csv", index=False)
    return ok

def require(name, ok, detail=""):
    if not check(name, ok, detail):
        raise RuntimeError(f"Stopped: {name}. {detail}")

TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
K = {"vsfc": 3, "vsmec": 7, "vihsd": 3, "xnli": 3, "vimmrc": 4}
CHANCE = {t: 1 / K[t] for t in TASKS}
VI_FAMILIES = {"SeaLLMs-v3", "VinaLLaMA"}
BIG_FAMILY = {"Qwen2.5-14B": ("Qwen2.5", 14.7), "Qwen3-14B": ("Qwen3", 14.8), "phi-4": ("Phi-4", 14.7)}
B = 2000          # item bootstrap, as in v4
B_WILD = 9999     # wild cluster bootstrap
SEED = 42

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#2b2a28",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "pdf.fonttype": 42,
})
FAM_COLOR = {"Qwen2.5": "#0d366b", "Qwen3": "#3987e5", "SeaLLMs-v3": "#1baf7a", "VinaLLaMA": "#eb6834", "bloomz": "#eda100",
             "Phi-3.5": "#6250d6", "Phi-4": "#a99ae8", "Mistral": "#8a8984", "Llama-3": "#c4314b"}
FIGS = []
def save(fig, name, caption):
    fig.savefig(f"{OUT}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT}/figures/{name}.pdf", bbox_inches="tight")
    FIGS.append((name, caption))
    plt.show()

def save_table(df, name, caption, floatfmt="%.3f"):
    df.to_csv(f"{OUT}/tables/{name}.csv")
    try:
        df.to_latex(f"{OUT}/tables/{name}.tex", float_format=floatfmt, caption=caption, label=f"tab:{name}", na_rep="--")
    except Exception as e:
        print("latex export failed:", name, repr(e)[:100])

def fmt_p(p):
    return "p < 1e-10" if p < 1e-10 else f"p = {p:.2g}"

def short(m):
    return m.split("/")[-1].replace("-Instruct", "").replace("-Chat", "").replace("-chat", "").replace("-mini-instruct", "-mini").replace("-v0.3", "")

def ci(a):
    return np.nanpercentile(a, 2.5), np.nanpercentile(a, 97.5)

def bal_acc(y, p, k):
    return np.mean([(p[y == c] == c).mean() for c in range(k) if (y == c).any()])

KEY, L = {}, ["# 1A pilot v7 · CPU re-analyses for review 1", ""]

S4 = pd.read_csv(f"{SRC4}/tables/summary_all.csv")
require("v4 summary has the strip condition and the validity flag", {"strip"} <= set(S4.cond) and "valid" in S4)

## B1 · Inference with few clusters

The size trend (v6) and the adapted-vs-general test (v4 follow-up F2) cluster standard errors by model. With 15–19 clusters the normal approximation gives p-values that are too small. Here every test is reported three ways:
1. **as published:** cluster-robust SE, normal reference;
2. **t(G−1):** the same SE with the small-sample factor G/(G−1)·(N−1)/(N−k) and a t distribution with G−1 degrees of freedom;
3. **wild cluster bootstrap (WCR):** the null is imposed (term set to 0), residuals are flipped per model with Webb's six-point weights, and the bootstrap t-statistics are compared with the observed one (9,999 draws).

With family fixed effects, a family with one model (Mistral, Llama-3, Phi-3.5, Phi-4) does not inform the size slope. The slope comes from Qwen2.5, Qwen3, SeaLLMs-v3, VinaLLaMA and bloomz; the 14B models enter through Qwen2.5-14B and Qwen3-14B only. So the Phi-3.5 → phi-4 confound the reviewer raises cannot drive the slope; the refit without Phi checks this.

In [ ]:
# per (model, task) normalised drop under strip: v4 for ≤ 8B, v6's 14B table for the 14B models
nd8 = S4[(S4.cond == "strip") & S4.valid].dropna(subset=["norm_drop"])[["model", "task", "family", "size", "norm_drop"]].copy()
nd8["model"] = nd8.model.map(short)
BIGT = pd.read_csv(f"{SRC6}/tables/table_big_models.csv")
nd14 = BIGT[BIGT.valid.astype(bool)][["model", "task", "nd_strip"]].rename(columns={"nd_strip": "norm_drop"})
nd14["family"] = nd14.model.map(lambda m: BIG_FAMILY[m][0])
nd14["size"] = nd14.model.map(lambda m: BIG_FAMILY[m][1])
ND = pd.concat([nd8, nd14], ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna(subset=["norm_drop"])
ND["log2size"] = np.log2(ND["size"])
ND["big"] = ND.model.isin(BIG_FAMILY)
ND["vi"] = ND.family.isin(VI_FAMILIES).astype(int)
print(len(ND), "pairs,", ND.model.nunique(), "models")

WEBB = np.array([-np.sqrt(1.5), -1, -np.sqrt(0.5), np.sqrt(0.5), 1, np.sqrt(1.5)])

def cluster_t(Xm, y, g, j):
    # OLS coefficient j and its CR1 t-statistic
    XtX_inv = np.linalg.pinv(Xm.T @ Xm)
    b = XtX_inv @ Xm.T @ y
    u = y - Xm @ b
    G, (N, k) = len(np.unique(g)), Xm.shape
    meat = np.zeros((k, k))
    for c in np.unique(g):
        s = Xm[g == c].T @ u[g == c]
        meat += np.outer(s, s)
    V = XtX_inv @ meat @ XtX_inv * G / (G - 1) * (N - 1) / (N - k)
    return b[j], b[j] / np.sqrt(V[j, j]), np.sqrt(V[j, j])

def three_ways(df, formula, term, label):
    df = df.reset_index(drop=True)
    g = pd.factorize(df.model)[0]
    G = len(np.unique(g))
    pub = smf.ols(formula, data=df).fit(cov_type="cluster", cov_kwds={"groups": g})
    ysr, Xd = patsy.dmatrices(formula, df, return_type="dataframe")
    j = list(Xd.columns).index(term)
    y, Xm = ysr.to_numpy().ravel(), Xd.to_numpy()
    b, t_obs, se = cluster_t(Xm, y, g, j)
    from scipy import stats
    p_t = 2 * stats.t.sf(abs(t_obs), G - 1)
    tcrit = stats.t.ppf(0.975, G - 1)
    # restricted fit (term = 0), then wild bootstrap of the t-statistic
    Xr = np.delete(Xm, j, axis=1)
    br = np.linalg.lstsq(Xr, y, rcond=None)[0]
    fit_r, u_r = Xr @ br, y - Xr @ br
    rng = np.random.default_rng(SEED)
    tb = np.empty(B_WILD)
    for i in range(B_WILD):
        w = WEBB[rng.integers(0, 6, G)][g]
        tb[i] = cluster_t(Xm, fit_r + w * u_r, g, j)[1]
    p_w = (np.abs(tb) >= abs(t_obs)).mean()
    return {"spec": label, "term": term, "coef": b, "lo (t)": b - tcrit * se, "hi (t)": b + tcrit * se,
            "p (normal, as published)": pub.pvalues[term], "p (t, G-1)": p_t, "p (wild cluster bootstrap)": p_w,
            "clusters": G, "pairs": len(df)}

SIZE_F = "norm_drop ~ log2size + C(task) + C(family)"
fam_sizes = ND.groupby("family").model.nunique()
rows = [three_ways(ND[~ND.big], SIZE_F, "log2size", "≤ 8B (v4 models)"),
        three_ways(ND, SIZE_F, "log2size", "all, with 14B"),
        three_ways(ND[~ND.family.str.startswith("Phi")], SIZE_F, "log2size", "with 14B, without Phi-3.5 and phi-4"),
        three_ways(ND[ND.family.isin(fam_sizes[fam_sizes >= 3].index)], SIZE_F, "log2size", "families with ≥ 3 sizes only")]
F2_F = "norm_drop ~ vi + log2size + C(task)"
rows += [three_ways(ND[~ND.big], F2_F, "vi", "adapted vs general, ≤ 8B (F2)"),
         three_ways(ND, F2_F, "vi", "adapted vs general, with 14B")]
INF = pd.DataFrame(rows).set_index(["spec", "term"])
save_table(INF, "table_b1_inference", "Size trend and adapted-vs-general: coefficient with t(G−1) CI, and p-values from the normal "
           "reference, t(G−1) and a wild cluster bootstrap (Webb weights, restricted).", "%.4g")
display(INF)

# reproduction of v6's and F2's published coefficients
SZ6 = pd.read_csv(f"{SRC6}/tables/table_size_trend.csv", index_col=0)
for lab6, lab in [("≤ 8B (v4)", "≤ 8B (v4 models)"), ("with 14B", "all, with 14B")]:
    if lab6 in SZ6.index:
        check(f"size slope reproduces v6 ({lab6})", abs(SZ6.loc[lab6, "slope_per_doubling"] - INF.loc[(lab, "log2size"), "coef"]) < 2e-3,
              f"v6 {SZ6.loc[lab6, 'slope_per_doubling']:.4f}, here {INF.loc[(lab, 'log2size'), 'coef']:.4f}")
KEY["b1"] = INF.reset_index().to_dict("records")

## B2 · A calibration prior estimated on the noisy inputs

v4's main predictions divide by a prior estimated on **clean dev** items. A deployed system that sees only unaccented input could instead estimate the prior on the inputs it receives (no labels needed). v4 computed this as `per-condition calibration` but only reported its mean drop. Here, for every valid ≤ 8B pair:
- accuracy under `strip` with the clean-dev prior (main) and with a prior from the stripped test inputs themselves;
- **recovery** of the per-condition prior = (acc_strip,percond − acc_strip,main) / (acc_clean − acc_strip,main), next to v5's restoration recovery on the same pair;
- label collapse: share of the most frequent predicted label under `strip`, and the number of pairs above 0.6.

In [ ]:
def softmax_rows(lp):
    lp = lp - lp.max(1, keepdims=True)
    e = np.exp(lp)
    return e / e.sum(1, keepdims=True)

R5 = pd.read_csv(f"{SRC5}/tables/table_restoration_per_model.csv")
R5["model"] = R5.model.map(short)
valid8 = S4[(S4.cond == "clean") & S4.valid][["model", "task"]].drop_duplicates()
rows = []
for m, t in valid8.itertuples(index=False):
    f = f"{SRC4}/preds/{t}__{m.replace('/', '__')}.csv"
    if not os.path.exists(f):
        continue
    P = pd.read_csv(f)
    lp = P[[f"lp{k}" for k in range(K[t])]].to_numpy(float)
    p = softmax_rows(lp)
    pc = np.empty(len(P), int)
    for c in P.cond.unique():
        i = (P.cond == c).to_numpy()
        pc[i] = (np.log(p[i]) - np.log(p[i].mean(0))).argmax(1)
    P["pred_percond"] = pc
    acc = lambda c, col: (P[P.cond == c][col] == P[P.cond == c].label).mean()
    top = lambda c, col: P[P.cond == c][col].value_counts(normalize=True).iloc[0]
    a0, a1, a1p, a0p = acc("clean", "pred"), acc("strip", "pred"), acc("strip", "pred_percond"), acc("clean", "pred_percond")
    lost = a0 - a1
    r5 = R5[(R5.model == short(m)) & (R5.task == t)]
    rows.append({"model": short(m), "task": t, "acc_clean": a0, "acc_strip": a1, "acc_strip_percond": a1p, "acc_clean_percond": a0p,
                 "nd_strip": (a0 - a1) / (a0 - CHANCE[t]), "nd_strip_percond": (a0p - a1p) / (a0p - CHANCE[t]),
                 "recovery_percond": (a1p - a1) / lost if lost > 0.05 else np.nan,
                 "recovery_restored": r5.recovery.iloc[0] if len(r5) else np.nan,
                 "top_share_strip": top("strip", "pred"), "top_share_strip_percond": top("strip", "pred_percond")})
NP = pd.DataFrame(rows)
require("per-condition calibration computed for every valid pair with a prediction file", len(NP) > 0, len(NP))
check("main strip accuracy matches v4's summary",
      np.allclose(NP.merge(S4.assign(model=S4.model.map(short))[S4.cond == "strip"][["model", "task", "acc"]], on=["model", "task"])
                  .eval("acc_strip - acc").abs().max(), 0, atol=1e-9))
by_task = NP.groupby("task")[["nd_strip", "nd_strip_percond", "recovery_percond", "recovery_restored",
                               "top_share_strip", "top_share_strip_percond"]].mean().reindex(TASKS)
by_task.loc["mean (pairs)"] = NP[by_task.columns].mean()
collapse = {"pairs": len(NP), "collapsed (top share > 0.6), main": int((NP.top_share_strip > 0.6).sum()),
            "collapsed (top share > 0.6), per-condition prior": int((NP.top_share_strip_percond > 0.6).sum())}
save_table(by_task, "table_b2_noisy_prior", "Strip with the clean-dev prior (main) vs a prior from the stripped test inputs: "
           "normalised drop, recovery, and share of the most frequent predicted label; restoration recovery (v5) on the same pairs.")
NP.to_csv(f"{OUT}/tables/table_b2_noisy_prior_per_pair.csv", index=False)
display(by_task.round(3)); print(collapse)
rng = np.random.default_rng(SEED)
rec = NP.recovery_percond.dropna().to_numpy()
boot = [rng.choice(rec, len(rec)).mean() for _ in range(B)]
KEY["b2"] = {"by_task": by_task.round(4).to_dict("index"), "collapse": collapse,
             "recovery_percond_mean": float(rec.mean()), "recovery_percond_ci_pairs": ci(boot)}

## B3 · Naturally unaccented vs accented comments

Balanced accuracy (mean recall over the three classes, so the class mix does not matter) per model on:
- `natural`: v6's 226 ViHSD test comments their authors wrote without diacritics;
- `natural, restored`: the same comments after the restorer;
- `accented, clean` and `accented, strip`: v4's 450 class-balanced ViHSD test comments, as written and stripped.

The two sets are different comments, so the bootstrap resamples each set on its own. A model that reads habitual unaccented writing well should score close to `accented, clean` on `natural`; one that does not should score close to `accented, strip`.

In [ ]:
rows, boots = [], []
NATT = pd.read_csv(f"{SRC6}/tables/table_natural.csv")
nat_models = set(NATT.model)
for f in sorted(os.listdir(f"{SRC6}/natural")):
    N = pd.read_csv(f"{SRC6}/natural/{f}")
    m = N.model.iloc[0]
    if short(m) not in nat_models:
        continue
    src = f"{SRC6}/big/preds/vihsd__{m.replace('/', '__')}.csv"
    src = src if os.path.exists(src) else f"{SRC4}/preds/vihsd__{m.replace('/', '__')}.csv"
    A = pd.read_csv(src)
    sets = {"natural": N[N.cond == "natural"], "natural, restored": N[N.cond == "natural_restored"],
            "accented, clean": A[A.cond == "clean"], "accented, strip": A[A.cond == "strip"]}
    r = {"model": short(m)}
    for k, d in sets.items():
        y, p = d.label.to_numpy(), d.pred.to_numpy()
        r[k] = bal_acc(y, p, 3)
        bs = [bal_acc(y[i], p[i], 3) for i in (rng.integers(0, len(y), len(y)) for _ in range(500))]
        r[f"{k} lo"], r[f"{k} hi"] = ci(bs)
    # where does natural sit between strip (0) and clean (1)?
    gap = r["accented, clean"] - r["accented, strip"]
    r["natural position (0 = strip, 1 = clean)"] = (r["natural"] - r["accented, strip"]) / gap if gap > 0.05 else np.nan
    rows.append(r)
NV = pd.DataFrame(rows).set_index("model")
check("B3 covers the models of v6's natural table", set(NV.index) == nat_models, f"{len(NV)} of {len(nat_models)}")
save_table(NV, "table_b3_natural_vs_accented", "ViHSD balanced accuracy: naturally unaccented comments (as written, restored) vs "
           "v4's accented comments (clean, stripped); 95% bootstrap CIs within each set.")
display(NV.round(3))
KEY["b3"] = {"means": NV[["natural", "natural, restored", "accented, clean", "accented, strip"]].mean().to_dict(),
             "position_median": float(NV["natural position (0 = strip, 1 = clean)"].median()),
             "models": len(NV)}

## B4 · Does strip → restore fall below clean for the 14B models?

v6 restored the stripped text for the 14B models. Here: accuracy after restoration minus clean accuracy, per valid pair, with a paired bootstrap over items. A negative value means the restorer's errors cost the model accuracy it had on the original text. (Applying the restorer to already accented text is a new condition and needs the GPU run, D5.)

In [ ]:
rows = []
for f in sorted(os.listdir(f"{SRC6}/big/preds")):
    P = pd.read_csv(f"{SRC6}/big/preds/{f}")
    m, t = P.model.iloc[0], P.task.iloc[0]
    ok = BIGT[(BIGT.model == short(m)) & (BIGT.task == t)].valid.astype(bool)
    if not (len(ok) and ok.iloc[0]) or "restored" not in set(P.cond):
        continue
    c = P[P.cond == "clean"].sort_values("item_id")
    r = P[P.cond == "restored"].sort_values("item_id")
    s = P[P.cond == "strip"].sort_values("item_id")
    yc, yr, ys = ((d.pred.to_numpy() == d.label.to_numpy()).astype(float) for d in (c, r, s))
    diff = [(yr[i] - yc[i]).mean() for i in (rng.integers(0, len(yc), len(yc)) for _ in range(B))]
    lo, hi = ci(diff)
    rows.append({"model": short(m), "task": t, "acc_clean": yc.mean(), "acc_strip": ys.mean(), "acc_restored": yr.mean(),
                 "restored − clean": yr.mean() - yc.mean(), "lo": lo, "hi": hi, "below clean (CI < 0)": hi < 0})
RB = pd.DataFrame(rows).set_index(["model", "task"])
save_table(RB, "table_b4_restored_vs_clean_14b", "14B models: accuracy after strip → restore minus clean accuracy (paired bootstrap 95% CI).")
display(RB.round(3))
KEY["b4"] = {"pairs": len(RB), "below_clean": int(RB["below clean (CI < 0)"].sum()),
             "mean_diff_by_model": RB.groupby("model")["restored − clean"].mean().to_dict()}

## B6 · Mechanism checks: Δ broken subset, ΔNLL per token, ambiguity

Inputs: v4's `tables/item_level.csv.gz` (one row per model × item × noisy condition, the model right on the clean item) and v4's model, an LPM with item × condition and model fixed effects, clustered by unit.
1. **Q11.** Refit without the tokenizer groups whose syllables split *less* after `strip` (Phi-3.5 and Mistral in v4; phi-4 is not in the item-level file).
2. **Q2.** ΔNLL per clean syllable (v4) vs ΔNLL per token, (NLL/tokens)_noisy − (NLL/tokens)_clean. The second changes when the token count changes, the first does not.
3. **M1.** The ambiguity of stripped words does not vary across models, so it is absorbed by item × condition effects. v4 estimated it in a separate model with model, task and condition effects only; that table is reported here as its own specification.

In [ ]:
X = pd.read_csv(f"{SRC4}/tables/item_level.csv.gz")
X = X.dropna(subset=["flip", "d_nll", "d_tok", "d_broken"]).copy()
X["d_nll_tok"] = X.nll / X.ntok.clip(lower=1) - X.nll_c / X.ntok_c.clip(lower=1)

def demean(df, cols, groups, iters=200, tol=1e-9):
    out = df[cols].astype(float).copy()
    for _ in range(iters):
        delta = 0
        for gcol in groups:
            mns = out.groupby(df[gcol].values).transform("mean")
            out -= mns
            delta = max(delta, float(np.abs(mns.to_numpy()).max()))
        if delta < tol:
            break
    return out

def fe(df, xs):
    df = df[df.item_cond.map(df.item_cond.value_counts()) > 1]
    if len(df) < 50 or df.model.nunique() < 2:
        return None
    dm = demean(df, ["flip"] + xs, ["item_cond", "model"])
    sd = df[xs].std().replace(0, np.nan)
    keep = [x for x in xs if np.isfinite(sd[x])]
    r = sm.OLS(dm.flip, dm[keep] / sd[keep]).fit(cov_type="cluster", cov_kwds={"groups": df.unit})
    return pd.DataFrame({"pp_per_SD": 100 * r.params, "lo": 100 * r.conf_int()[0], "hi": 100 * r.conf_int()[1], "p": r.pvalues,
                         "rows": len(df)})

dir_by_tok = X[X.cond == "strip"].groupby("tok_class").d_broken.mean()
fewer = list(dir_by_tok[dir_by_tok < 0].index)
fam_of = X.groupby("tok_class").family.agg(lambda s: ", ".join(sorted(set(s))))
print("tokenizer groups that split fewer syllables after strip:", {g: fam_of[g] for g in fewer})
same_cls = X[X.family == "Qwen2.5"].tok_class.mode().iloc[0]
rows = []
def add(name, df, xs):
    r = fe(df, xs)
    if r is not None:
        for k in r.index:
            rows.append({"sample / spec": name, "term": k, **r.loc[k].to_dict()})
add("all valid models (v4)", X, ["d_nll", "d_tok", "d_broken"])
add("without groups that split fewer syllables", X[~X.tok_class.isin(fewer)], ["d_nll", "d_tok", "d_broken"])
add("without them, Δ broken alone", X[~X.tok_class.isin(fewer)], ["d_broken"])
add("ΔNLL per token instead of per syllable", X, ["d_nll_tok", "d_tok", "d_broken"])
add("same tokenizer, ΔNLL per syllable (v4)", X[X.tok_class == same_cls], ["d_nll"])
add("same tokenizer, ΔNLL per token", X[X.tok_class == same_cls], ["d_nll_tok"])
MECH = pd.DataFrame(rows).set_index(["sample / spec", "term"])
FE4 = pd.read_csv(f"{SRC4}/tables/table_item_level_fe.csv")
v4b = FE4[(FE4["sample"] == "all valid models") & (FE4.term == "d_broken")].pp_per_SD
check("v4's Δ broken coefficient is reproduced", len(v4b) == 1 and abs(MECH.loc[("all valid models (v4)", "d_broken"), "pp_per_SD"] - v4b.iloc[0]) < 1e-6)
save_table(MECH, "table_b6_mechanism", "Same-item fixed-effects LPM (pp per SD): without the tokenizers whose syllables split less after "
           "stripping, and with ΔNLL per token.")
AMB = pd.read_csv(f"{SRC4}/tables/table_ambiguity.csv", index_col=0)
AMB.index = AMB.index.str.replace("z_", "")
save_table(AMB, "table_b6_ambiguity_spec", "Ambiguity model (v4): flip ~ ambiguity + ΔNLL + Δtokens + model, task and condition "
           "fixed effects (no item effects), partial and strip conditions, clustered by unit.")
display(MECH.round(4)); display(AMB.round(4))
KEY["b6"] = {"fewer_groups": {g: fam_of[g] for g in fewer}, "mechanism": MECH.reset_index().to_dict("records"),
             "ambiguity_spec": AMB.reset_index().to_dict("records")}

## B7 · Figures for the paper, from the final data

- **Dose-response:** mean normalised drop over valid models for 25/50/75% (mean of the three seeds, v5) and 100% (`strip`), per task; teen code and mixed as separate markers at the share of words they change.
- **Size:** mean normalised drop under `strip` per model against parameters (log scale), one line per family, ≤ 8B from v4 and 14B from v6.

In [ ]:
SS = pd.read_csv(f"{SRC5}/tables/table_seed_stability.csv")
MD = pd.read_csv(f"{SRC4}/tables/table_mean_drop.csv")
TNAME = {"vsfc": "VSFC", "vsmec": "VSMEC", "vihsd": "ViHSD", "xnli": "XNLI", "vimmrc": "ViMMRC"}
TCOL = {"vsfc": "#0d366b", "vsmec": "#3987e5", "vihsd": "#eb6834", "xnli": "#1baf7a", "vimmrc": "#6250d6"}
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
for t in TASKS:
    d = SS[SS.task == t].sort_values("level")
    s = MD[(MD.task == t) & (MD.cond == "strip")]
    x = [0] + list(d.level) + [100]
    y = [0] + list(d["mean of seeds"]) + list(s.mean_norm_drop)
    lo = [0] + list(d.lo) + list(s.lo)
    hi = [0] + list(d.hi) + list(s.hi)
    ax.plot(x, y, marker="o", color=TCOL[t], label=TNAME[t])
    ax.fill_between(x, lo, hi, color=TCOL[t], alpha=0.12, lw=0)
ax.set(xlabel="share of accented syllables stripped (%)", ylabel="normalised drop", xticks=[0, 25, 50, 75, 100], xlim=(-2, 102))
ax.legend(fontsize=7, ncol=2)
ax.set_title("a. Dose-response (3-seed mean, 95% CI)", loc="left")
ax = axes[1]
x = np.arange(len(TASKS))
for j, (c, col) in enumerate([("strip", "#2b2a28"), ("teencode", "#eb6834"), ("mixed", "#6250d6")]):
    d = MD[MD.cond == c].set_index("task").reindex(TASKS)
    ax.bar(x + (j - 1) * 0.26, d.mean_norm_drop, 0.24, color=col, label=c,
           yerr=[d.mean_norm_drop - d.lo, d.hi - d.mean_norm_drop], error_kw={"lw": 0.8, "ecolor": "#52514e"})
ax.set_xticks(x, [TNAME[t] for t in TASKS], fontsize=7)
ax.set(ylabel="normalised drop")
ax.legend(fontsize=7)
ax.set_title("b. Strip vs teen code vs mixed", loc="left")
plt.tight_layout()
save(fig, "fig_dose_final", "Mean normalised drop over valid models up to 8B (95% CI). (a) By share of stripped syllables; partial levels "
     "average three noise seeds. (b) Full stripping, teen code, and teen code followed by 50% stripping.")

fig, ax = plt.subplots(figsize=(3.6, 2.8))
mm = ND.groupby(["model", "family", "size"]).norm_drop.mean().reset_index()
for fam, g in mm.groupby("family"):
    g = g.sort_values("size")
    ax.plot(g["size"], g.norm_drop, marker="o", color=FAM_COLOR.get(fam, "#52514e"), lw=1.5 if len(g) > 1 else 0, label=fam)
ax.set(xscale="log", xlabel="parameters (B, log scale)", ylabel="mean normalised drop, strip")
ax.set_xticks([1.5, 3, 7, 14]); ax.set_xticklabels(["1.5", "3", "7", "14"])
ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.legend(fontsize=6.5, ncol=2)
plt.tight_layout()
save(fig, "fig_size_final", "Mean normalised drop under full stripping per model (valid tasks) against size; one line per family. "
     "Models up to 8B from v4, 14B models from v6.")

## C · Sample for manual diacritization

All 270 naturally unaccented ViHSD test comments (v6, before the ≥ 20% filter), with their labels from the ViHSD test split. The sample is stratified by class: every `offensive` and `hate` comment up to 30 each, and `clean` comments for the rest, 100 in total, in random order.

Columns: `id`, `label`, `in_v6_set` (passed the ≥ 20% filter), `restored_share`, `as_written`, `restorer` (the machine output), and **`manual`**, which starts as a copy of `restorer`. Edit only `manual` so that it is the comment written with correct diacritics, keeping the same words (do not fix slang or spelling). Leave a word as it is when it has no correct diacritized form (English, names, slang). Mark `unsure` = 1 for comments you cannot decide.

The file is written to Drive only: ViHSD contains offensive content and its licence does not allow redistribution in a public repository.

In [ ]:
NR = pd.read_csv(f"{SRC6}/data/natural_restored.csv", keep_default_na=False)
NI = pd.read_csv(f"{SRC6}/data/natural_items.csv", keep_default_na=False)
def nfc(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()
PARQUET = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/{}/0000.parquet"
try:
    raw = pd.read_parquet(PARQUET.format("ura-hcmut/UIT-ViHSD", "default", "test"))
    raw["free_text"] = raw.free_text.map(nfc)
    lab = raw.drop_duplicates("free_text").set_index("free_text").label_id
    NR["label"] = NR.natural.map(lab)
except Exception as e:
    print("download failed, labels only for the 226 used items:", repr(e)[:200])
    NR["label"] = NR.natural.map(NI[NI.cond == "natural"].drop_duplicates("text").set_index("text").label)
def changed_share(a, b):
    w = [(x, y) for x, y in zip(a.split(" "), b.split(" ")) if any(c.isalpha() for c in x)]
    return sum(x != y for x, y in w) / max(len(w), 1)
NR["restored_share"] = [changed_share(a, b) for a, b in zip(NR.natural, NR.restored)]
NR["in_v6_set"] = NR.natural.isin(set(NI[NI.cond == "natural"].text)).astype(int)
check("labels found for every natural comment", NR.label.notna().all(), f"{NR.label.notna().sum()} of {len(NR)}")
check("the v6 set is the ≥ 20% subset", (NR.in_v6_set == (NR.restored_share >= 0.2)).all() or SMOKE)
NR = NR.dropna(subset=["label"])
NR["label"] = NR.label.astype(int)
LABELS = {0: "clean", 1: "offensive", 2: "hate"}
rng_c = np.random.default_rng(SEED)
n_total = min(100, len(NR))
take = {1: min(30, (NR.label == 1).sum()), 2: min(30, (NR.label == 2).sum())}
take[0] = n_total - take[1] - take[2]
parts = [NR[NR.label == k].sample(n=min(n, (NR.label == k).sum()), random_state=SEED) for k, n in take.items()]
CS = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
CS = pd.DataFrame({"id": range(1, len(CS) + 1), "label": CS.label.map(LABELS), "in_v6_set": CS.in_v6_set,
                   "restored_share": CS.restored_share.round(2), "as_written": CS.natural, "restorer": CS.restored,
                   "manual": CS.restored, "unsure": 0})
CS.to_csv(f"{OUT}/data/natural_sample_100.csv", index=False, encoding="utf-8-sig")
display(CS.label.value_counts().to_frame("comments"))
display(pd.crosstab(CS.label, CS.in_v6_set))
KEY["c"] = {"pool": len(NR), "sample": len(CS), "by_class": CS.label.value_counts().to_dict(),
            "in_v6_set": int(CS.in_v6_set.sum())}

## C2 · Restorer error and label check on the annotated sample

Input: the sample from C after manual annotation, with two extra columns, `label_suggested` (filled only where the annotator disagrees with the ViHSD label) and `label_note`. Put it on Drive as `1A-pilot-v7-cpu/data/natural_sample_100_annotated.csv` (in the Sheet: File → Download → CSV). If that file is missing, on Colab the cell exports the Google Sheet named `natural_sample_100` through the Drive API, which asks you to sign in once.

1. **Restorer word error.** For the comments in v6's 226, word by word against `manual`, counting only words with letters, case ignored. Reported with and without the comments marked `unsure`; 95% bootstrap CI over comments.
2. **Labels.** The ViHSD label against the annotator's, by class, and the share of the 226 that would be disputed, weighting each class by its size in the 226 (the sample is stratified, so the raw share in the sample is not that number).
3. **B3 re-scored.** Balanced accuracy on `natural` and `natural, restored` per model: (a) as in B3; (b) the disputed comments dropped; (c) the annotator's label where given. Then the same on the reviewed comments only (ViHSD labels, then annotator labels). One person re-annotated, knowing the hypothesis, so (b)–(e) are a sensitivity analysis; the main result stays (a).

In [ ]:
ANN_PATH = f"{OUT}/data/natural_sample_100_annotated.csv"
if not os.path.exists(ANN_PATH) and IN_COLAB:
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    drv = build("drive", "v3")
    hits = drv.files().list(q="name = 'natural_sample_100' and mimeType = 'application/vnd.google-apps.spreadsheet' and trashed = false",
                            fields="files(id, name, modifiedTime)").execute()["files"]
    require("exactly one Google Sheet named natural_sample_100", len(hits) == 1, f"{len(hits)} found")
    open(ANN_PATH, "wb").write(drv.files().export(fileId=hits[0]["id"], mimeType="text/csv").execute())
    print("exported the Sheet, last modified", hits[0]["modifiedTime"])
require("annotated sample found", os.path.exists(ANN_PATH), ANN_PATH)
AN = pd.read_csv(ANN_PATH, keep_default_na=False, dtype=str)
need = {"id", "label", "in_v6_set", "as_written", "restorer", "manual", "unsure", "label_suggested"}
require("annotated sample has the needed columns", need <= set(AN.columns), sorted(need - set(AN.columns)))
for c in ["as_written", "restorer", "manual"]:
    AN[c] = AN[c].map(nfc)
AN["in_v6"] = AN.in_v6_set.str.strip() == "1"
AN["unsure"] = AN.unsure.str.strip() == "1"
AN["label_suggested"] = AN.label_suggested.str.strip().str.lower()
check("suggested labels are clean / offensive / hate", AN.label_suggested.isin(["", *LABELS.values()]).all(),
      sorted(set(AN.label_suggested) - {"", *LABELS.values()}))
AN["disputed"] = (AN.label_suggested != "") & (AN.label_suggested != AN.label)

def strip_vi(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    return unicodedata.normalize("NFC", "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"))
bad = AN[[strip_vi(m).lower() != w.lower() for m, w in zip(AN.manual, AN.as_written)]]
check("manual only adds diacritics (stripping it gives back the comment)", bad.empty, ", ".join(bad.id.head(10)))

# 1 · restorer word error against the manual diacritization
def word_errors(a, b):
    a, b = a.lower().split(), b.lower().split()
    if len(a) != len(b):
        return None
    w = [(x, y) for x, y in zip(a, b) if any(ch.isalpha() for ch in y)]
    return sum(x != y for x, y in w), len(w)
AN["werr"] = [word_errors(a, b) for a, b in zip(AN.restorer, AN.manual)]
check("restorer and manual have the same words", AN.werr.notna().all(), ", ".join(AN.id[AN.werr.isna()]))
rows = []
for name, d in {"v6's 226, all": AN[AN.in_v6], "v6's 226, without unsure": AN[AN.in_v6 & ~AN.unsure],
                "all 100": AN}.items():
    e = np.array([x[0] for x in d.werr.dropna()]); n = np.array([x[1] for x in d.werr.dropna()])
    bs = [(e[i].sum() / n[i].sum()) for i in (rng_c.integers(0, len(e), len(e)) for _ in range(B))]
    lo, hi = ci(bs)
    rows.append({"set": name, "comments": len(e), "words": int(n.sum()), "wrong words": int(e.sum()),
                 "word error": e.sum() / n.sum(), "lo": lo, "hi": hi, "comments with ≥ 1 error": float((e > 0).mean())})
RE = pd.DataFrame(rows).set_index("set")
save_table(RE, "table_c2_restorer_error", "Restorer word error against a manual diacritization of naturally unaccented ViHSD comments; "
           "95% bootstrap CI over comments.")
display(RE.round(3))

# 2 · labels
XT = pd.crosstab(AN.label, AN.label_suggested.where(AN.label_suggested != "", AN.label).rename("annotator"))
display(XT)
NAT_I = NI[NI.cond == "natural"].drop_duplicates("text")
size226 = NAT_I.label.map(LABELS).value_counts()
by = AN[AN.in_v6].groupby("label").agg(reviewed=("id", "size"), disputed=("disputed", "sum"))
by["share disputed"] = by.disputed / by.reviewed
by["in v6's 226"] = size226.reindex(by.index).fillna(0).astype(int)
by["estimated disputed in 226"] = by["share disputed"] * by["in v6's 226"]
save_table(by, "table_c2_label_dispute", "ViHSD labels disputed by the annotator, by ViHSD class, among the sampled comments of v6's 226, "
           "and the implied number in the 226.")
display(by.round(3))

# 3 · B3 re-scored
ann_by_text = AN[AN.in_v6].drop_duplicates("as_written").set_index("as_written")
NAT_I = NAT_I.assign(text=NAT_I.text.map(nfc))
NAT_I["reviewed"] = NAT_I.text.isin(ann_by_text.index)
NAT_I["disputed"] = NAT_I.text.map(ann_by_text.disputed).fillna(False).astype(bool)
inv = {v: k for k, v in LABELS.items()}
NAT_I["label_ann"] = [inv[ann_by_text.label_suggested[t]] if d else l for t, d, l in zip(NAT_I.text, NAT_I.disputed, NAT_I.label)]
check("every sampled comment of v6's 226 is found among v6's natural items", NAT_I.reviewed.sum() == AN.in_v6.sum(),
      f"{NAT_I.reviewed.sum()} of {AN.in_v6.sum()}")
meta = NAT_I.set_index("item_id")[["reviewed", "disputed", "label_ann"]]
VARIANTS = {"(a) all, ViHSD labels": lambda d: (d, "label"), "(b) all, disputed dropped": lambda d: (d[~d.disputed], "label"),
            "(c) all, annotator labels": lambda d: (d, "label_ann"), "(d) reviewed, ViHSD labels": lambda d: (d[d.reviewed], "label"),
            "(e) reviewed, annotator labels": lambda d: (d[d.reviewed], "label_ann")}
rows = []
for f in sorted(os.listdir(f"{SRC6}/natural")):
    N = pd.read_csv(f"{SRC6}/natural/{f}")
    m = short(N.model.iloc[0])
    if m not in NV.index:
        continue
    for cond, cname in [("natural", "natural"), ("natural_restored", "natural, restored")]:
        d = N[N.cond == cond].join(meta, on="item_id")
        for v, fn in VARIANTS.items():
            dd, col = fn(d)
            rows.append({"model": m, "set": cname, "variant": v, "items": len(dd),
                         "bal acc": bal_acc(dd[col].to_numpy(), dd.pred.to_numpy(), 3)})
RS = pd.DataFrame(rows)
check("C2 re-scores every model of B3", RS.model.nunique() == len(NV), f"{RS.model.nunique()} of {len(NV)}")
RSW = RS.pivot_table(index=["set", "model"], columns="variant", values="bal acc")
save_table(RSW, "table_c2_b3_rescored", "B3 balanced accuracy on naturally unaccented comments, re-scored with the annotator's labels.")
display(RSW.round(3))
ref = NV[["accented, clean", "accented, strip"]].mean()
MEANS = RS.groupby(["set", "variant"])["bal acc"].mean().unstack("set")
MEANS["items"] = RS[RS.set == "natural"].groupby("variant")["items"].first()
nat_pos = RS[RS.set == "natural"].join(NV[["accented, clean", "accented, strip"]], on="model")
gap = nat_pos["accented, clean"] - nat_pos["accented, strip"]
nat_pos["position"] = np.where(gap > 0.05, (nat_pos["bal acc"] - nat_pos["accented, strip"]) / gap, np.nan)
MEANS["natural position, median"] = nat_pos.groupby("variant").position.median()
save_table(MEANS, "table_c2_b3_means", "Mean balanced accuracy over models for each re-scoring of B3, and the median position of "
           f"`natural` between accented strip (0) and accented clean (1); accented means: clean {ref.iloc[0]:.3f}, strip {ref.iloc[1]:.3f}.")
display(MEANS.round(3))
KEY["c2"] = {"restorer": RE.round(4).to_dict("index"), "dispute": by.round(4).to_dict("index"),
             "disputed_reviewed": int(AN[AN.in_v6].disputed.sum()), "reviewed_in_226": int(AN.in_v6.sum()),
             "transitions": {f"{a}→{b}": int(n) for (a, b), n in AN[AN.disputed].groupby(["label", "label_suggested"]).size().items()},
             "b3_means": MEANS.round(4).to_dict("index"), "accented": ref.round(4).to_dict()}

## Conclusions (generated from the numbers above)

In [ ]:
L += ["## B1 · Few clusters", "| Spec | Term | Coef [t(G−1) 95% CI] | p normal | p t(G−1) | p wild | Clusters |", "|---|---|---|---|---|---|---|"]
for (s, term), r in INF.iterrows():
    L.append(f"| {s} | {term} | {r['coef']:+.3f} [{r['lo (t)']:+.3f}, {r['hi (t)']:+.3f}] | {r['p (normal, as published)']:.2g} | "
             f"{r['p (t, G-1)']:.2g} | {r['p (wild cluster bootstrap)']:.2g} | {int(r['clusters'])} |")
b2 = KEY["b2"]
L += ["", "## B2 · Prior from the noisy inputs",
      f"- Over {b2['collapse']['pairs']} valid pairs, a prior estimated on the stripped inputs recovers {100 * b2['recovery_percond_mean']:.0f}% "
      f"[{100 * b2['recovery_percond_ci_pairs'][0]:.0f}, {100 * b2['recovery_percond_ci_pairs'][1]:.0f}] of the loss; restoration recovers "
      f"{100 * NP.recovery_restored.mean():.0f}% on the same pairs.",
      f"- Pairs whose most frequent predicted label exceeds 60% under strip: {b2['collapse']['collapsed (top share > 0.6), main']} with the clean-dev prior, "
      f"{b2['collapse']['collapsed (top share > 0.6), per-condition prior']} with the noisy prior."]
b3 = KEY["b3"]["means"]
L += ["", "## B3 · Natural vs accented ViHSD (mean balanced accuracy over models)",
      f"- natural {b3['natural']:.3f}, natural restored {b3['natural, restored']:.3f}, accented clean {b3['accented, clean']:.3f}, "
      f"accented strip {b3['accented, strip']:.3f}. Median position of natural between strip (0) and clean (1): {KEY['b3']['position_median']:.2f}."]
b4 = KEY["b4"]
L += ["", "## B4 · 14B: strip → restore vs clean",
      f"- {b4['below_clean']} of {b4['pairs']} valid pairs fall significantly below clean. Mean difference by model: "
      + ", ".join(f"{m} {v:+.3f}" for m, v in b4["mean_diff_by_model"].items()) + "."]
L += ["", "## B6 · Mechanism", f"- Tokenizer groups that split fewer syllables after strip: {KEY['b6']['fewer_groups']}."]
for (s, term), r in MECH.iterrows():
    L.append(f"- {s} · {term}: {r['pp_per_SD']:+.2f} pp/SD [{r['lo']:+.2f}, {r['hi']:+.2f}], {fmt_p(r['p'])}")
L += ["", "Ambiguity model (no item effects):"] + [f"- {k}: {r['pp_per_SD']:+.2f} pp/SD, {fmt_p(r['p'])}" for k, r in AMB.iterrows()]
L += ["", "## C · Manual sample", f"- {KEY['c']['sample']} of {KEY['c']['pool']} comments: {KEY['c']['by_class']}; "
      f"{KEY['c']['in_v6_set']} are in v6's 226. File: `data/natural_sample_100.csv`."]
c2 = KEY["c2"]
L += ["", "## C2 · Annotated sample"]
for k, r in c2["restorer"].items():
    L.append(f"- Restorer word error, {k}: {100 * r['word error']:.1f}% [{100 * r['lo']:.1f}, {100 * r['hi']:.1f}] of {r['words']} words "
             f"in {r['comments']} comments; {100 * r['comments with ≥ 1 error']:.0f}% of comments have at least one wrong word.")
L.append(f"- Labels disputed: {c2['disputed_reviewed']} of {c2['reviewed_in_226']} reviewed comments of v6's 226 ("
         + ", ".join(f"{k} {v}" for k, v in c2["transitions"].items()) + "); by ViHSD class: "
         + ", ".join(f"{k} {int(r['disputed'])}/{int(r['reviewed'])}" for k, r in c2["dispute"].items())
         + f"; implied ≈ {sum(r['estimated disputed in 226'] for r in c2['dispute'].values()):.0f} of the 226.")
L += ["", "| Re-scoring | items | natural | natural, restored | natural position (median) |", "|---|---|---|---|---|"]
for v, r in c2["b3_means"].items():
    L.append(f"| {v} | {int(r['items'])} | {r['natural']:.3f} | {r['natural, restored']:.3f} | {r['natural position, median']:.2f} |")
L.append(f"- Accented reference (mean over models): clean {c2['accented']['accented, clean']:.3f}, strip {c2['accented']['accented, strip']:.3f}.")
L += ["", "## Figures"]
L += [f"- `{n}`: {c}" for n, c in FIGS]
open(f"{OUT}/conclusions.md", "w").write("\n".join(L))
json.dump(KEY, open(f"{OUT}/key_numbers.json", "w"), indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("\n".join(L))
failed = [c for c in CHECKS if not c["ok"]]
print(f"\n{len(CHECKS) - len(failed)}/{len(CHECKS)} checks passed")